In [1]:
from __future__ import annotations

import re
from collections import Counter
from pathlib import Path
from typing import Any

import arcpy
from arcpy import env

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# =============================================================================
# CONFIG
# =============================================================================

ROUNDSTAMP = "2026-06"
BASE_DIR = Path(r"D:\A02-Projects\Shapefile_Cleaning")

RAW_WORKSPACE = BASE_DIR / "00_raw" / ROUNDSTAMP

DRY_RUN = False
# True  = count report only, do not drop fields
# False = count first, then actually drop target fields

COUNT_REPORT = (
    BASE_DIR
    / "05_reports"
    / f"{ROUNDSTAMP}"
    / f"05_{ROUNDSTAMP}-PARCEL_TYPE.xlsx"
)
COUNT_REPORT.parent.mkdir(parents=True, exist_ok=True)


# =============================================================================
# FIELD CONFIG
# =============================================================================

PARCEL_TYPE_FIELD = "PARCEL_TYP"

FIELDS_TO_DROP = [
    "APID",
    "INSIDE_X",
    "INSIDE_Y",
]

# Expected shapefile name pattern:
#     <branch_code>_<zone>.shp
# Example:
#     50000000_47.shp
SHP_NAME_PATTERN = re.compile(r"^(?P<branch_code>\d+)_(?P<zone>47|48)$", re.IGNORECASE)


# =============================================================================
# BASIC HELPERS
# =============================================================================

def normalize_text(value: Any) -> str:
    """
    Convert value to clean text.
    """
    if value is None:
        return ""

    text = str(value).replace("\u00a0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_group_value(value: Any) -> str:
    """
    Normalize PARCEL_TYP value for grouping.

    Examples:
        None    -> "NULL"
        ""      -> "NULL"
        1       -> "1"
        1.0     -> "1"
        "01"    -> "01"
        " 1 "   -> "1"
    """
    if value is None:
        return "NULL"

    text = normalize_text(value)

    if text == "":
        return "NULL"

    # Keep leading zeros if the original text has them, e.g. "01".
    if re.fullmatch(r"0\d+", text):
        return text

    # Convert numeric-looking values like 1.0 to 1.
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except Exception:
        pass

    return text


def sort_group_values(values: list[str] | set[str]) -> list[str]:
    """
    Natural-ish sorting for PARCEL_TYP values.

    Order:
        numeric values first, then NULL, then text values.
    """
    def sort_key(value: str) -> tuple[int, float, str]:
        if value == "NULL":
            return (1, 0, value)

        try:
            return (0, float(value), value)
        except Exception:
            return (2, 0, value)

    return sorted(values, key=sort_key)


def parse_shapefile_name(shp_path: Path) -> tuple[str, str]:
    """
    Parse branch_code and zone from shapefile name.

    Returns:
        branch_code, zone

    If filename does not match expected pattern:
        "", ""
    """
    match = SHP_NAME_PATTERN.fullmatch(shp_path.stem)

    if not match:
        return "", ""

    return match.group("branch_code"), match.group("zone")


def find_shapefiles(root_dir: Path) -> list[Path]:
    """
    Find every .shp file under RAW_WORKSPACE, including subfolders.
    """
    if not root_dir.exists():
        raise FileNotFoundError(f"RAW_WORKSPACE not found: {root_dir}")

    return sorted(
        path
        for path in root_dir.rglob("*")
        if path.is_file() and path.suffix.lower() == ".shp"
    )


def get_field_map(feature_class: Path | str) -> dict[str, arcpy.Field]:
    """
    Return field map using uppercase field names.
    """
    return {
        field.name.upper(): field
        for field in arcpy.ListFields(str(feature_class))
    }


def get_actual_field_name(feature_class: Path | str, field_name: str) -> str | None:
    """
    Return actual field name, case-insensitive.

    Example:
        requested: PARCEL_TYP
        actual:    PARCEL_TYP
    """
    field_map = get_field_map(feature_class)
    field_obj = field_map.get(field_name.upper())

    if field_obj is None:
        return None

    return field_obj.name


def get_feature_count(feature_class: Path | str) -> int:
    """
    Return feature count.
    """
    return int(arcpy.management.GetCount(str(feature_class))[0])


# =============================================================================
# COUNT LOGIC
# =============================================================================

def count_parcel_type_one_shapefile(shp_path: Path) -> tuple[dict[str, Any], list[dict[str, Any]], dict[str, int]]:
    """
    Count polygons/features grouped by PARCEL_TYP for one shapefile.

    Returns:
        summary_row:
            One row for Shapefile Summary sheet.

        detail_rows:
            One row per PARCEL_TYP value for Parcel Type sheet.

        counter:
            PARCEL_TYP -> count, used for Pivot sheet.
    """
    shapefile_name = shp_path.stem
    branch_code, zone = parse_shapefile_name(shp_path)

    summary_row = {
        "shapefile name": shapefile_name,
        "full path": str(shp_path),
        "branch_code": branch_code,
        "zone": zone,
        "total polygon": 0,
        "distinct PARCEL_TYP count": 0,
        "status": "",
        "message": "",
    }

    detail_rows: list[dict[str, Any]] = []
    counter: Counter[str] = Counter()

    try:
        if not arcpy.Exists(str(shp_path)):
            summary_row["status"] = "ERROR"
            summary_row["message"] = "Shapefile does not exist or ArcPy cannot read it"
            return summary_row, detail_rows, dict(counter)

        actual_parcel_type_field = get_actual_field_name(shp_path, PARCEL_TYPE_FIELD)

        if actual_parcel_type_field is None:
            total_count = get_feature_count(shp_path)

            summary_row["total polygon"] = total_count
            summary_row["status"] = "ERROR"
            summary_row["message"] = f"Field not found: {PARCEL_TYPE_FIELD}"
            return summary_row, detail_rows, dict(counter)

        with arcpy.da.SearchCursor(str(shp_path), [actual_parcel_type_field]) as cursor:
            for row in cursor:
                parcel_type = normalize_group_value(row[0])
                counter[parcel_type] += 1

        total_count = sum(counter.values())
        distinct_count = len(counter)

        summary_row["total polygon"] = total_count
        summary_row["distinct PARCEL_TYP count"] = distinct_count

        if total_count == 0:
            summary_row["status"] = "EMPTY"
            summary_row["message"] = "Shapefile has no polygon/features"
        else:
            summary_row["status"] = "SUCCESS"
            summary_row["message"] = "Count completed"

        for parcel_type in sort_group_values(set(counter.keys())):
            detail_rows.append(
                {
                    "shapefile name": shapefile_name,
                    "full path": str(shp_path),
                    "branch_code": branch_code,
                    "zone": zone,
                    "PARCEL_TYP": parcel_type,
                    "Count": counter[parcel_type],
                }
            )

        return summary_row, detail_rows, dict(counter)

    except PermissionError:
        summary_row["status"] = "ERROR"
        summary_row["message"] = (
            "Permission denied. File may be open in ArcGIS Pro, QGIS, "
            "or Windows Explorer preview."
        )
        return summary_row, detail_rows, dict(counter)

    except Exception as exc:
        summary_row["status"] = "ERROR"
        summary_row["message"] = repr(exc)
        return summary_row, detail_rows, dict(counter)



# =============================================================================
# DROP FIELD LOGIC
# =============================================================================

def get_existing_drop_fields(feature_class: Path | str, target_fields: list[str]) -> list[str]:
    """
    Return actual existing field names that should be dropped.

    The check is case-insensitive, but the returned names use the actual
    shapefile field names.
    """
    field_map = get_field_map(feature_class)

    existing_fields: list[str] = []

    for field_name in target_fields:
        field_obj = field_map.get(field_name.upper())

        if field_obj is not None:
            existing_fields.append(field_obj.name)

    return existing_fields


def drop_fields_one_shapefile(shp_path: Path) -> dict[str, Any]:
    """
    Drop APID, INSIDE_X, and INSIDE_Y after parcel-type counting.

    The count is done before this function runs, so the dropped fields do not
    affect the parcel-type report.
    """
    result = {
        "shapefile name": shp_path.stem,
        "full path": str(shp_path),
        "APID exists": "",
        "INSIDE_X exists": "",
        "INSIDE_Y exists": "",
        "fields to drop": "",
        "count of drop": 0,
        "status": "",
        "message": "",
    }

    try:
        if not arcpy.Exists(str(shp_path)):
            result["status"] = "ERROR"
            result["message"] = "Shapefile does not exist or ArcPy cannot read it"
            return result

        field_map = get_field_map(shp_path)

        for field_name in FIELDS_TO_DROP:
            result[f"{field_name} exists"] = "YES" if field_name.upper() in field_map else "NO"

        existing_drop_fields = get_existing_drop_fields(
            feature_class=shp_path,
            target_fields=FIELDS_TO_DROP,
        )

        result["fields to drop"] = ", ".join(existing_drop_fields)
        result["count of drop"] = len(existing_drop_fields)

        if not existing_drop_fields:
            result["status"] = "SKIP"
            result["message"] = "None of the target fields exist"
            return result

        if DRY_RUN:
            result["status"] = "DRY_RUN"
            result["message"] = "Checked only, fields not dropped"
            return result

        arcpy.management.DeleteField(
            in_table=str(shp_path),
            drop_field=existing_drop_fields,
        )

        result["status"] = "DROPPED"
        result["message"] = "Fields dropped successfully"
        return result

    except PermissionError:
        result["status"] = "ERROR"
        result["message"] = (
            "Permission denied. File may be open in ArcGIS Pro, QGIS, "
            "or Windows Explorer preview."
        )
        return result

    except Exception as exc:
        result["status"] = "ERROR"
        result["message"] = repr(exc)
        return result


# =============================================================================
# EXCEL REPORT
# =============================================================================

def style_worksheet(ws) -> None:
    """
    Apply basic Excel formatting.
    """
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)

    thin_border = Border(
        left=Side(style="thin", color="D9E2F3"),
        right=Side(style="thin", color="D9E2F3"),
        top=Side(style="thin", color="D9E2F3"),
        bottom=Side(style="thin", color="D9E2F3"),
    )

    if ws.max_row >= 1:
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = thin_border

    for row in ws.iter_rows():
        for cell in row:
            cell.border = thin_border
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions


def autosize_columns(ws, max_width: int = 100) -> None:
    """
    Auto-size Excel columns with a sensible maximum width.
    """
    for column_cells in ws.columns:
        max_length = 0
        column_letter = get_column_letter(column_cells[0].column)

        for cell in column_cells:
            value = cell.value

            if value is None:
                continue

            max_length = max(max_length, len(str(value)))

        ws.column_dimensions[column_letter].width = min(max_length + 2, max_width)


def write_sheet(ws, headers: list[str], rows: list[dict[str, Any]]) -> None:
    """
    Write a dict-row table to one worksheet.
    """
    ws.append(headers)

    for row in rows:
        ws.append([row.get(header, "") for header in headers])

    style_worksheet(ws)
    autosize_columns(ws)


def build_pivot_rows(
    summary_rows: list[dict[str, Any]],
    count_by_shapefile: dict[str, dict[str, int]],
    all_parcel_type_values: list[str],
) -> tuple[list[str], list[dict[str, Any]]]:
    """
    Build wide-format pivot rows:
        one shapefile per row,
        one PARCEL_TYP value per count column.
    """
    headers = [
        "shapefile name",
        "full path",
        "branch_code",
        "zone",
        "total polygon",
    ] + [f"PARCEL_TYP={value}" for value in all_parcel_type_values]

    pivot_rows: list[dict[str, Any]] = []

    for summary in summary_rows:
        shapefile_name = summary.get("shapefile name", "")
        counter = count_by_shapefile.get(shapefile_name, {})

        row = {
            "shapefile name": shapefile_name,
            "full path": summary.get("full path", ""),
            "branch_code": summary.get("branch_code", ""),
            "zone": summary.get("zone", ""),
            "total polygon": summary.get("total polygon", 0),
        }

        for parcel_type in all_parcel_type_values:
            row[f"PARCEL_TYP={parcel_type}"] = counter.get(parcel_type, 0)

        pivot_rows.append(row)

    return headers, pivot_rows


def write_report_excel(
    detail_rows: list[dict[str, Any]],
    summary_rows: list[dict[str, Any]],
    count_by_shapefile: dict[str, dict[str, int]],
    all_parcel_type_values: list[str],
    drop_rows: list[dict[str, Any]],
) -> None:
    """
    Write Excel report to COUNT_REPORT.
    """
    wb = Workbook()

    # -------------------------------------------------------------------------
    # Sheet 1: Parcel Type
    # -------------------------------------------------------------------------
    ws_detail = wb.active
    ws_detail.title = "Parcel Type"

    write_sheet(
        ws=ws_detail,
        headers=[
            "shapefile name",
            "full path",
            "branch_code",
            "zone",
            "PARCEL_TYP",
            "Count",
        ],
        rows=detail_rows,
    )

    # -------------------------------------------------------------------------
    # Sheet 2: Shapefile Summary
    # -------------------------------------------------------------------------
    ws_summary_table = wb.create_sheet("Shapefile Summary")

    write_sheet(
        ws=ws_summary_table,
        headers=[
            "shapefile name",
            "full path",
            "branch_code",
            "zone",
            "total polygon",
            "distinct PARCEL_TYP count",
            "status",
            "message",
        ],
        rows=summary_rows,
    )

    # -------------------------------------------------------------------------
    # Sheet 3: Pivot
    # -------------------------------------------------------------------------
    ws_pivot = wb.create_sheet("Pivot")
    pivot_headers, pivot_rows = build_pivot_rows(
        summary_rows=summary_rows,
        count_by_shapefile=count_by_shapefile,
        all_parcel_type_values=all_parcel_type_values,
    )

    write_sheet(
        ws=ws_pivot,
        headers=pivot_headers,
        rows=pivot_rows,
    )

    # -------------------------------------------------------------------------
    # Sheet 4: Drop Field
    # -------------------------------------------------------------------------
    ws_drop = wb.create_sheet("drop_field")

    write_sheet(
        ws=ws_drop,
        headers=[
            "shapefile name",
            "full path",
            "APID exists",
            "INSIDE_X exists",
            "INSIDE_Y exists",
            "fields to drop",
            "count of drop",
            "status",
            "message",
        ],
        rows=drop_rows,
    )

    # -------------------------------------------------------------------------
    # Sheet 5: Summary
    # -------------------------------------------------------------------------
    ws_summary = wb.create_sheet("Summary")
    ws_summary.append(["Item", "Value"])

    total_shapefiles = len(summary_rows)
    total_polygon = sum(int(row.get("total polygon") or 0) for row in summary_rows)

    summary_info_rows = [
        ["ROUNDSTAMP", ROUNDSTAMP],
        ["RAW_WORKSPACE", str(RAW_WORKSPACE)],
        ["DRY_RUN", DRY_RUN],
        ["COUNT_REPORT", str(COUNT_REPORT)],
        ["Total shapefiles checked", total_shapefiles],
        ["Total polygon/features", total_polygon],
        ["Distinct PARCEL_TYP values", len(all_parcel_type_values)],
        ["Count SUCCESS", sum(1 for row in summary_rows if row.get("status") == "SUCCESS")],
        ["Count EMPTY", sum(1 for row in summary_rows if row.get("status") == "EMPTY")],
        ["Count ERROR", sum(1 for row in summary_rows if row.get("status") == "ERROR")],
        ["Drop DROPPED", sum(1 for row in drop_rows if row.get("status") == "DROPPED")],
        ["Drop DRY_RUN", sum(1 for row in drop_rows if row.get("status") == "DRY_RUN")],
        ["Drop SKIP", sum(1 for row in drop_rows if row.get("status") == "SKIP")],
        ["Drop ERROR", sum(1 for row in drop_rows if row.get("status") == "ERROR")],
        [
            "Total fields dropped / would drop",
            sum(int(row.get("count of drop") or 0) for row in drop_rows),
        ],
    ]

    for row in summary_info_rows:
        ws_summary.append(row)

    style_worksheet(ws_summary)
    autosize_columns(ws_summary)

    wb.save(COUNT_REPORT)


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    env.overwriteOutput = True

    print("============================================================")
    print("Count shapefile polygons grouped by PARCEL_TYP, then drop fields")
    print("============================================================")
    print(f"ROUNDSTAMP    : {ROUNDSTAMP}")
    print(f"RAW_WORKSPACE : {RAW_WORKSPACE}")
    print(f"DRY_RUN       : {DRY_RUN}")
    print(f"COUNT_REPORT  : {COUNT_REPORT}")
    print("------------------------------------------------------------")

    shapefiles = find_shapefiles(RAW_WORKSPACE)

    print(f"Shapefiles found: {len(shapefiles):,}")
    print("------------------------------------------------------------")

    detail_rows: list[dict[str, Any]] = []
    summary_rows: list[dict[str, Any]] = []
    count_by_shapefile: dict[str, dict[str, int]] = {}
    all_parcel_type_values: set[str] = set()
    drop_rows: list[dict[str, Any]] = []

    for idx, shp_path in enumerate(shapefiles, start=1):
        print(f"[{idx:,}/{len(shapefiles):,}] {shp_path}")

        summary_row, shp_detail_rows, counter = count_parcel_type_one_shapefile(shp_path)

        summary_rows.append(summary_row)
        detail_rows.extend(shp_detail_rows)
        count_by_shapefile[shp_path.stem] = counter
        all_parcel_type_values.update(counter.keys())

        print(
            f"  COUNT | {summary_row.get('status', '')} | "
            f"total={summary_row.get('total polygon', 0)} | "
            f"distinct={summary_row.get('distinct PARCEL_TYP count', 0)} | "
            f"{summary_row.get('message', '')}"
        )

        drop_row = drop_fields_one_shapefile(shp_path)
        drop_rows.append(drop_row)

        print(
            f"  DROP  | {drop_row.get('status', '')} | "
            f"drop={drop_row.get('count of drop', 0)} | "
            f"{drop_row.get('message', '')}"
        )

    sorted_parcel_type_values = sort_group_values(all_parcel_type_values)

    write_report_excel(
        detail_rows=detail_rows,
        summary_rows=summary_rows,
        count_by_shapefile=count_by_shapefile,
        all_parcel_type_values=sorted_parcel_type_values,
        drop_rows=drop_rows,
    )

    print("============================================================")
    print(f"Report written: {COUNT_REPORT}")
    print("Processing completed.")


if __name__ == "__main__":
    main()


Count shapefile polygons grouped by PARCEL_TYP, then drop fields
ROUNDSTAMP    : 2026-06
RAW_WORKSPACE : D:\A02-Projects\Shapefile_Cleaning\00_raw\2026-06
DRY_RUN       : False
COUNT_REPORT  : D:\A02-Projects\Shapefile_Cleaning\05_reports\2026-06\05_2026-06-PARCEL_TYPE.xlsx
------------------------------------------------------------
Shapefiles found: 12
------------------------------------------------------------
[1/12] D:\A02-Projects\Shapefile_Cleaning\00_raw\2026-06\36000000\36000000_47.shp
  COUNT | SUCCESS | total=854 | distinct=10 | Count completed
  DROP  | DROPPED | drop=3 | Fields dropped successfully
[2/12] D:\A02-Projects\Shapefile_Cleaning\00_raw\2026-06\36000000\36000000_48.shp
  COUNT | SUCCESS | total=3492 | distinct=10 | Count completed
  DROP  | DROPPED | drop=3 | Fields dropped successfully
[3/12] D:\A02-Projects\Shapefile_Cleaning\00_raw\2026-06\36010000\36010000_47.shp
  COUNT | SUCCESS | total=448 | distinct=9 | Count completed
  DROP  | DROPPED | drop=3 | Fields 